# Lab 4: Clinical Vision — Chest X-Ray Pneumonia Detection
## CCI Session 4

**Duration:** 15 minutes  
**GPU Required:** T4 GPU  

### Clinical Scenario
> KHCC's radiology department processes hundreds of chest X-rays daily. Automated pneumonia screening could flag high-priority cases for immediate review. You'll test a pre-trained image model on chest X-rays, then fine-tune it to detect pneumonia.

### Objective
- Load and visualize a chest X-ray dataset
- Test a pre-trained model on medical images (see it fail)
- Fine-tune MobileNetV2 for pneumonia detection
- Evaluate with clinical metrics (precision, recall, confusion matrix)

---
## Setup
First, install dependencies and verify GPU access.

In [ ]:
!pip install -q datasets torchvision matplotlib scikit-learn

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only!'}")
print(f"PyTorch version: {torch.__version__}")

---
## Cell 1: Load & Visualize Chest X-Ray Data

The dataset contains **5,863 chest X-ray images** with two classes:
- **NORMAL (0):** Healthy lungs
- **PNEUMONIA (1):** Signs of pneumonia

Dataset: `hf-vision/chest-xray-pneumonia` (CC-BY-4.0 license)

In [ ]:
# === CELL 1: LOAD CHEST X-RAY DATA ===
from datasets import load_dataset
import matplotlib.pyplot as plt

# TODO: Load the chest X-ray pneumonia dataset
# ds = load_dataset("hf-vision/chest-xray-pneumonia")

# TODO: Print dataset splits and sizes
# print(ds)

# TODO: Print label distribution for the training split
# Hint: Count how many images have label 0 vs label 1
# labels = [ex['label'] for ex in ds['train']]
# print(f"NORMAL: {labels.count(0)}, PNEUMONIA: {labels.count(1)}")

# TODO: Display a grid of 8 sample images (4 NORMAL, 4 PNEUMONIA)
# Use matplotlib to show the images with their labels
# Hint: Filter by label, pick 4 from each, plot in a 2x4 grid
# label_names = ['NORMAL', 'PNEUMONIA']
#
# fig, axes = plt.subplots(2, 4, figsize=(16, 8))
# for class_idx in range(2):
#     class_examples = [ex for ex in ds['train'] if ex['label'] == class_idx]
#     for i in range(4):
#         ax = axes[class_idx][i]
#         ax.imshow(class_examples[i]['image'], cmap='gray')
#         ax.set_title(label_names[class_idx], fontsize=12)
#         ax.axis('off')
# plt.suptitle('Chest X-Ray Samples — Can YOU tell which ones have pneumonia?', fontsize=14)
# plt.tight_layout()
# plt.show()

---
## Cell 2: Data Preprocessing

MobileNetV2 expects:
- **224x224** pixel images
- **3-channel RGB** (some X-rays are grayscale)
- **ImageNet normalization** (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

In [ ]:
# === CELL 2: PREPROCESSING ===
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

# TODO: Define image transforms:
# - Resize to 224x224 (MobileNet input size)
# - Convert to RGB (some X-rays are grayscale)
# - Convert to tensor
# - Normalize with ImageNet mean/std

transform = transforms.Compose([
    # TODO: Add transforms here
    # transforms.Resize((224, 224)),
    # transforms.ToTensor(),
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# TODO: Create a custom Dataset class that:
# 1. Takes the HF dataset split
# 2. Applies transforms to each image
# 3. Returns (image_tensor, label)
#
# class CXRDataset(Dataset):
#     def __init__(self, hf_dataset, transform=None):
#         self.dataset = hf_dataset
#         self.transform = transform
#
#     def __len__(self):
#         return len(self.dataset)
#
#     def __getitem__(self, idx):
#         item = self.dataset[idx]
#         image = item['image'].convert('RGB')  # Ensure 3-channel
#         label = item['label']
#         if self.transform:
#             image = self.transform(image)
#         return image, label

# TODO: Create DataLoaders for train and test splits
# train_dataset = CXRDataset(ds['train'], transform)
# test_dataset = CXRDataset(ds['test'], transform)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
# print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

---
## Cell 3: Load Pre-trained MobileNetV2

MobileNetV2 was trained on **ImageNet** (1.2M images, 1000 classes like cats, dogs, cars).  
It has **never seen a chest X-ray** before. Let's see what happens!

In [ ]:
# === CELL 3: LOAD MOBILENET ===
from torchvision import models
import torch.nn as nn

# TODO: Load MobileNetV2 pre-trained on ImageNet
# model = models.mobilenet_v2(pretrained=True)

# TODO: Print the model architecture — notice the final classifier layer
# print(model.classifier)
# It has 1000 outputs (ImageNet classes) — we need 2 (NORMAL/PNEUMONIA)

# TODO: Move model to GPU
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = model.to(device)
# print(f"Model loaded on: {device}")

---
## Cell 4: Baseline — Test Before Fine-Tuning

Let's see how a model trained on everyday photos performs on medical images.  
**Prediction:** It should perform at roughly **random chance (~50%)** since it has no concept of pneumonia.

In [ ]:
# === CELL 4: BASELINE — BEFORE FINE-TUNING ===

# TODO: Run 20 test images through the pre-trained model
# The model outputs 1000 ImageNet class probabilities — pick the top one
#
# model.eval()
# correct = 0
# total = 0
#
# with torch.no_grad():
#     for images, labels in test_loader:
#         images, labels = images.to(device), labels.to(device)
#         outputs = model(images)
#         # The model has 1000 outputs — we can't directly compare to our 2 classes
#         # Let's just see what ImageNet class it predicts
#         _, predicted = torch.max(outputs, 1)
#         total += labels.size(0)
#         if total >= 20:
#             break
#
# # Show what the model thinks X-rays are (ImageNet labels)
# # It will predict things like 'envelope', 'monitor', etc. — NOT pneumonia!
# print("The model was trained on ImageNet — it has no idea what a chest X-ray is!")
# print("Before fine-tuning, accuracy on our binary task is essentially random (~50%)")

---
## Cell 5: Modify Model for Binary Classification

We need to:
1. **Replace** the final layer (1000 ImageNet classes -> 2 classes)
2. **Freeze** early layers (keep learned features, only train the new classifier)

This is called **transfer learning** — reusing knowledge from one domain in another.

In [ ]:
# === CELL 5: MODIFY MODEL ===

# TODO: Replace the final classifier layer for binary classification
# model.classifier[1] = nn.Linear(model.last_channel, 2)

# TODO: Freeze early layers (feature extractor) — only train the classifier
# for param in model.features.parameters():
#     param.requires_grad = False

# TODO: Move updated model to GPU
# model = model.to(device)

# TODO: Count trainable vs frozen parameters
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total_params = sum(p.numel() for p in model.parameters())
# print(f"Trainable parameters: {trainable:,} / {total_params:,} ({100*trainable/total_params:.1f}%)")

---
## Cell 6: Fine-Tune the Model

Now we train **only the classifier head** on our chest X-ray data.  
With a T4 GPU, 3 epochs should take about 3-5 minutes.

In [ ]:
# === CELL 6: FINE-TUNE ===
import torch.optim as optim

# TODO: Define loss function and optimizer
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# TODO: Training loop (3 epochs)
# num_epochs = 3
#
# for epoch in range(num_epochs):
#     model.train()
#     running_loss = 0.0
#     correct = 0
#     total = 0
#
#     for batch_idx, (images, labels) in enumerate(train_loader):
#         images, labels = images.to(device), labels.to(device)
#
#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#
#         running_loss += loss.item()
#         _, predicted = torch.max(outputs, 1)
#         total += labels.size(0)
#         correct += (predicted == labels).sum().item()
#
#         if (batch_idx + 1) % 50 == 0:
#             print(f"  Batch {batch_idx+1}/{len(train_loader)} — Loss: {loss.item():.4f}")
#
#     epoch_loss = running_loss / len(train_loader)
#     epoch_acc = 100 * correct / total
#     print(f"Epoch [{epoch+1}/{num_epochs}] — Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.1f}%")
#
# print("\nTraining complete!")

---
## Cell 7: Evaluation — Clinical Metrics

In clinical settings, we care about:
- **Recall (Sensitivity):** Of all actual pneumonia cases, how many did we catch?
- **Precision:** Of all cases we flagged as pneumonia, how many actually were?
- **Confusion Matrix:** See exactly where the model makes mistakes

**Key question:** In pneumonia screening, is recall or precision more important?

In [ ]:
# === CELL 7: EVALUATION ===
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# TODO: Run the fine-tuned model on the test set
# Collect predictions and true labels
#
# model.eval()
# all_preds = []
# all_labels = []
#
# with torch.no_grad():
#     for images, labels in test_loader:
#         images = images.to(device)
#         outputs = model(images)
#         _, predicted = torch.max(outputs, 1)
#         all_preds.extend(predicted.cpu().numpy())
#         all_labels.extend(labels.numpy())

# TODO: Print classification report (precision, recall, F1 for each class)
# label_names = ['NORMAL', 'PNEUMONIA']
# print("Classification Report:")
# print(classification_report(all_labels, all_preds, target_names=label_names))

# TODO: Print confusion matrix
# cm = confusion_matrix(all_labels, all_preds)
# print("Confusion Matrix:")
# print(f"                Predicted NORMAL  Predicted PNEUMONIA")
# print(f"Actual NORMAL        {cm[0][0]:>5}            {cm[0][1]:>5}")
# print(f"Actual PNEUMONIA     {cm[1][0]:>5}            {cm[1][1]:>5}")

# TODO: Discuss:
# In a pneumonia screening context, is recall or precision more important?
# What's the clinical cost of a false negative (missing pneumonia)?
# What's the clinical cost of a false positive (unnecessary follow-up)?

---
## Cell 8: Before vs After Comparison

Let's visualize the dramatic improvement from fine-tuning.

In [ ]:
# === CELL 8: BEFORE VS AFTER ===

# TODO: Show side-by-side comparison table
# print("="*55)
# print(f"{'Metric':<25} {'Before (ImageNet)':<18} {'After (Fine-tuned)'}")
# print("="*55)
# print(f"{'Accuracy':<25} {'~50% (random)':<18} {'~85-90%'}")
# print(f"{'Recall (Pneumonia)':<25} {'~0% (no concept)':<18} {'~90%+'}")
# print(f"{'Precision (Pneumonia)':<25} {'N/A':<18} {'~85%+'}")
# print("="*55)

# TODO: Display 6 test images with:
# - True label
# - Model prediction (after fine-tuning)
# - Correct/Incorrect indicator
#
# fig, axes = plt.subplots(2, 3, figsize=(15, 10))
# label_names = ['NORMAL', 'PNEUMONIA']
#
# for i, ax in enumerate(axes.flat):
#     image, label = test_dataset[i]
#     # Run prediction
#     with torch.no_grad():
#         output = model(image.unsqueeze(0).to(device))
#         _, pred = torch.max(output, 1)
#         pred = pred.item()
#
#     # Display image (undo normalization for visualization)
#     img = image.permute(1, 2, 0).numpy()
#     img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # Undo normalize
#     img = np.clip(img, 0, 1)
#     ax.imshow(img)
#
#     correct = "✓" if pred == label else "✗"
#     color = 'green' if pred == label else 'red'
#     ax.set_title(f"True: {label_names[label]}\nPred: {label_names[pred]} {correct}",
#                  fontsize=11, color=color)
#     ax.axis('off')
#
# plt.suptitle('Fine-Tuned Model Predictions on Test Images', fontsize=14)
# plt.tight_layout()
# plt.show()

---
## Stretch Challenge: Unfreeze Feature Layers

We only trained the classifier head. What if we also fine-tune the last 2 feature blocks?  
This lets the model adapt its learned features to the medical imaging domain.

**Try it:** Unfreeze the last 2 feature blocks and retrain — does accuracy improve?

In [ ]:
# === STRETCH CHALLENGE ===

# TODO: Unfreeze the last 2 feature blocks of MobileNetV2
# MobileNetV2 has 19 feature blocks (indices 0-18)
# Unfreeze blocks 17 and 18:
#
# for param in model.features[17:].parameters():
#     param.requires_grad = True

# TODO: Use a lower learning rate for the unfrozen feature layers
# optimizer = optim.Adam([
#     {'params': model.features[17:].parameters(), 'lr': 1e-4},
#     {'params': model.classifier.parameters(), 'lr': 1e-3},
# ])

# TODO: Train for 2 more epochs and compare accuracy
# Does unfreezing the feature layers improve performance?

---
## KHCC Connection

Automated chest X-ray screening is being evaluated for KHCC's emergency department triage workflow. Models like the one you just built could:

- **Flag urgent cases** for immediate radiologist review
- **Reduce turnaround time** for pneumonia detection
- **Support resource-limited settings** where radiologists are scarce

**Key takeaway:** A pre-trained model knows nothing about medicine — but with just a few minutes of fine-tuning on labeled clinical data, it can become a useful screening tool.

---
*Lab 4 Complete — CCI Session 4*